# Informações das bases de imagens

Este notebook reúne três bases: `dirty`, `imagens de rua limpa` e `Lixo - Noite`. A célula seguinte conta os arquivos de imagem e as imagens rotuladas em cada pasta.

Nas bases `dirty` e `Lixo - Noite`, uma imagem é considerada rotulada quando há um arquivo JSON com o mesmo nome-base. Em `imagens de rua limpa`, a categoria é indicada pela própria pasta; por isso, todas as imagens dessa pasta são contabilizadas como rotuladas, embora não haja arquivos JSON.

In [6]:
from pathlib import Path

PASTA_BASE = Path.cwd()
EXTENSOES_IMAGEM = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff", ".avif"}

bases = [
    {
        "nome": "dirty",
        "pasta_imagens": PASTA_BASE / "dirty",
        "pasta_rotulos": PASTA_BASE / "dirty" / "labels",
    },
    {
        "nome": "imagens de rua limpa",
        "pasta_imagens": PASTA_BASE / "imagens_classificadas_ruas_limpas",
        "pasta_rotulos": None,
    },
    {
        "nome": "Lixo - Noite",
        "pasta_imagens": PASTA_BASE / "Lixo - Noite",
        "pasta_rotulos": PASTA_BASE / "Lixo - Noite" / "labels_2",
    },
]

resumo = []
for base in bases:
    pasta_imagens = base["pasta_imagens"]
    pasta_rotulos = base["pasta_rotulos"]
    if not pasta_imagens.is_dir():
        raise FileNotFoundError(f"Pasta de imagens não encontrada: {pasta_imagens}")

    imagens = [
        arquivo
        for arquivo in pasta_imagens.rglob("*")
        if arquivo.is_file() and arquivo.suffix.casefold() in EXTENSOES_IMAGEM
    ]

    if pasta_rotulos is None:
        quantidade_rotulos = 0
        quantidade_rotuladas = len(imagens)
        criterio = "rotuladas pela categoria da pasta"
    else:
        if not pasta_rotulos.is_dir():
            raise FileNotFoundError(f"Pasta de rótulos não encontrada: {pasta_rotulos}")
        arquivos_rotulos = [arquivo for arquivo in pasta_rotulos.rglob("*.json") if arquivo.is_file()]
        nomes_rotulados = {arquivo.stem.casefold() for arquivo in arquivos_rotulos}
        quantidade_rotulos = len(arquivos_rotulos)
        quantidade_rotuladas = sum(
            imagem.stem.casefold() in nomes_rotulados for imagem in imagens
        )
        criterio = "rotuladas com JSON de mesmo nome"

    resumo.append(
        (base["nome"], len(imagens), quantidade_rotulos, quantidade_rotuladas, criterio)
    )

print("RESUMO DAS BASES DE IMAGENS")
print(f"{'Base':<26} {'Imagens':>8} {'Arquivos JSON':>14} {'Imagens rotuladas':>19}")
print("-" * 70)
for nome, quantidade_imagens, quantidade_rotulos, quantidade_rotuladas, criterio in resumo:
    print(f"{nome:<26} {quantidade_imagens:>8} {quantidade_rotulos:>14} {quantidade_rotuladas:>19}")
    print(f"  Critério: {criterio}")

print("-" * 70)
print(f"Total de imagens: {sum(item[1] for item in resumo)}")
print(f"Total de imagens rotuladas: {sum(item[3] for item in resumo)}")

RESUMO DAS BASES DE IMAGENS
Base                        Imagens  Arquivos JSON   Imagens rotuladas
----------------------------------------------------------------------
dirty                           103            103                 103
  Critério: rotuladas com JSON de mesmo nome
imagens de rua limpa             32              0                  32
  Critério: rotuladas pela categoria da pasta
Lixo - Noite                     11             11                  11
  Critério: rotuladas com JSON de mesmo nome
----------------------------------------------------------------------
Total de imagens: 146
Total de imagens rotuladas: 146


In [5]:
import filecmp
import shutil

PASTA_CONSOLIDADA = PASTA_BASE / "imagens_rotuladas_consolidadas"
PASTA_LABELS_CONSOLIDADA = PASTA_CONSOLIDADA / "labels"

pares_rotulados = []
for base in bases:
    if base["nome"] not in {"dirty", "Lixo - Noite"}:
        continue

    pasta_imagens = base["pasta_imagens"]
    pasta_rotulos = base["pasta_rotulos"]
    if not pasta_rotulos.is_dir():
        raise FileNotFoundError(f"Pasta de rótulos não encontrada: {pasta_rotulos}")

    rotulos_por_nome = {}
    for rotulo in pasta_rotulos.rglob("*.json"):
        if not rotulo.is_file():
            continue
        chave = rotulo.stem.casefold()
        if chave in rotulos_por_nome:
            raise ValueError(f"Há mais de um JSON para o nome-base {rotulo.stem!r}")
        rotulos_por_nome[chave] = rotulo

    for imagem in pasta_imagens.rglob("*"):
        if not imagem.is_file() or imagem.suffix.casefold() not in EXTENSOES_IMAGEM:
            continue
        rotulo = rotulos_por_nome.get(imagem.stem.casefold())
        if rotulo is not None:
            pares_rotulados.append((imagem, rotulo))

nomes_imagens = [imagem.name.casefold() for imagem, _ in pares_rotulados]
nomes_rotulos = [rotulo.name.casefold() for _, rotulo in pares_rotulados]
if len(nomes_imagens) != len(set(nomes_imagens)):
    raise ValueError("Existem nomes de imagem duplicados entre as bases de origem.")
if len(nomes_rotulos) != len(set(nomes_rotulos)):
    raise ValueError("Existem nomes de JSON duplicados entre as bases de origem.")
if not pares_rotulados:
    raise ValueError("Nenhum par de imagem e JSON foi encontrado.")

PASTA_CONSOLIDADA.mkdir(parents=True, exist_ok=True)
PASTA_LABELS_CONSOLIDADA.mkdir(parents=True, exist_ok=True)

imagens_copiadas = 0
rotulos_copiados = 0
for imagem, rotulo in pares_rotulados:
    destino_imagem = PASTA_CONSOLIDADA / imagem.name
    destino_rotulo = PASTA_LABELS_CONSOLIDADA / rotulo.name

    if destino_imagem.exists():
        if not filecmp.cmp(imagem, destino_imagem, shallow=False):
            raise FileExistsError(f"Já existe uma imagem diferente no destino: {destino_imagem}")
    else:
        shutil.copy2(imagem, destino_imagem)
        imagens_copiadas += 1

    if destino_rotulo.exists():
        if not filecmp.cmp(rotulo, destino_rotulo, shallow=False):
            raise FileExistsError(f"Já existe um JSON diferente no destino: {destino_rotulo}")
    else:
        shutil.copy2(rotulo, destino_rotulo)
        rotulos_copiados += 1

print(f"Pasta consolidada: {PASTA_CONSOLIDADA}")
print(f"Pares imagem/JSON encontrados: {len(pares_rotulados)}")
print(f"Novas imagens copiadas: {imagens_copiadas}")
print(f"Novos JSONs copiados: {rotulos_copiados}")
print("As imagens e os rótulos das pastas originais foram mantidos.")

Pasta consolidada: c:\Users\alexa\Downloads\Teste Base de Imagens TCC\imagens_rotuladas_consolidadas
Pares imagem/JSON encontrados: 114
Novas imagens copiadas: 0
Novos JSONs copiados: 0
As imagens e os rótulos das pastas originais foram mantidos.
